# <font color="#418FDE" size="10" uppercase>**D: TensorFlow Config**</font>
----

> Last update: 20240103

By the end of this lecture, you will be able to:
* Investigate TensorFlow layers & del configuration information.
* Apply TensorFlow gradient tape to train ML models.


## **1. TensorFlow Config**

> TensorFlow is known for its flexibility & wide range of functionalities, which often require specific configurations to optimize performance for different tasks & hardware environments. These configurations can include settings for computational graphs, session executions, hardware utilization, & model hyperparameters.

> One key aspect of TensorFlow's configurability is its ability to manage computational resources. TensorFlow allows users to configure how much memory to allocate to specific operations & how to distribute computations across available hardware resources, such as CPUs, GPUs, & TPUs (Tensor Processing Units). This is particularly important for optimizing performance & efficiency, especially when dealing with large-scale ML models or computationally intensive tasks. The framework provides various tools & settings for this purpose, such as the tf.ConfigProto class, which allows for detailed configuration of TensorFlow sessions, including aspects like GPU memory allocation & the enabling of JIT (Just-In-Time) compilation.

> Another important configuration aspect in TensorFlow is the setup of model hyperparameters & training configurations. TensorFlow offers extensive options for configuring the various components of ML models, including layers, activation functions, optimizers, & regularization techniques. This flexibility is crucial for fine-tuning models to achieve the best performance on specific tasks. Moreover, TensorFlow's configuration settings extend to aspects like data input pipelines, serialization & restoration of models, & integration with high-level APIs like Keras. While this configurability makes TensorFlow a powerful tool for ML practitioners, it also introduces a level of complexity, requiring users to have a good understanding of the available options & their implications on model performance & resource utilization.

### **1.1. Layer Configs**

In [ ]:
#@title Example of Layer Config - Create A Sample Model
'''
Runtime: CPU
tf.keras.Input:                       https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Conv2D:               https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:         https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.Model:                       https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                   https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.models                       https://www.tensorflow.org/api_docs/python/tf/keras/models
tf.keras.models.clone_model:          https://www.tensorflow.org/api_docs/python/tf/keras/models/clone_model
'''
import tensorflow as tf

# Define the LeNet model architecture using the functional API
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(6, (5, 5))(inputs)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Conv2D(16, (5, 5))(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(120)(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.Dense(84)(x)
x = tf.keras.layers.ReLU()(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="model")

# Compile the model with optimizer, loss function, & metrics
model.compile(optimizer=tf.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()

In [ ]:
#@title Example of Layer Config - Layer Names
layer_names  = [layer.name for layer in model.layers]
print(layer_names )

In [ ]:
#@title Example of Layer Config - Layer Classes
df  = {layer.name: layer.__class__.__name__ for layer in model.layers}
for i0, key in enumerate(df):
    print(f"{i0+1:03d}. {key:-<{50}}{str(df[key]):->{50}}")

In [ ]:
#@title Example of Layer Config - Layer Input Tensors
# Create an empty dictionary named 'df' to store information about each layer in a model.
# The keys will be the layer names, & the values will be lists of input layer names.
df = {}

# Iterate through each layer in the 'model' object.
# The 'model' object is assumed to be a predefined model containing multiple layers.
for layer in model.layers:

    # Check if the layer's input attribute is a list, which indicates multiple inputs.
    # This is typical for layers in complex models that merge inputs from previous layers.
    if isinstance(layer.input, list):

        # If there are multiple inputs, extract the names of each input layer.
        # The input names are obtained from the 'name' attribute &
        # split at the '/' character to remove additional sub-layer information,
        # keeping only the main layer name.
        layer_inputs_case = [inp.name.split('/')[0] for inp in layer.input]
    else:
        # If the input is not a list, it means the layer has a single input.
        # Extract the name of this single input layer using the same method as above.
        layer_inputs_case = [layer.input.name.split('/')[0]]

    # Assign the extracted list of input layer names to the current layer's name
    # in the 'df' dictionary. This creates a mapping of each layer to its input layers.
    df[layer.name] = layer_inputs_case


for i0, key in enumerate(df):
    print(f"{i0+1:03d}. {key:-<{50}}{str(df[key]):->{50}}")

In [ ]:
#@title Example of Layer Config - Layer Input Tensors Shapes
df = {}
for layer in model.layers:
    if isinstance(layer.input, list):
        layer_inputs_shape = [inp.shape for inp in layer.input]
    else:
        layer_inputs_shape = [layer.input.shape]
    df[layer.name] = layer_inputs_shape

for i0, key in enumerate(df):
    print(f"{i0+1:03d}. {key:-<{50}}{str(df[key]):->{50}}")

In [ ]:
#@title Example of Layer Config - Layer Trainability
df = {layer.name: layer.trainable for layer in model.layers}
for i0, key in enumerate(df):
    print(f"{i0+1:03d}. {key:-<{50}}{str(df[key]):->{50}}")

In [ ]:
#@title Example of Layer Config - Layer Weights
df = {layer.name: [(weight.name, weight.numpy().shape, weight.trainable) for weight in layer.weights] for layer in model.layers}
for i0, key in enumerate(df):
    print(f"{i0+1:03d}. {key:-<{50}}{str(df[key]):->{50}}")


In [ ]:
#@title Example of Layer Config - Layer Get Config
for layer in model.layers:
    print(layer.get_config())


In [ ]:
#@title Example of Layer Config - New Layer with Get Config
# Create a new convolutional layer, 'layer_new', by copying the configuration of the second layer
# (index 1) of a pre-defined model ('model'). The get_config() method of a layer returns a
# configuration dictionary which can be used to create a new layer with the same configuration.
layer_new = tf.keras.layers.Conv2D(**model.layers[1].get_config())

# Print the configuration of the newly created layer.
# The configuration includes details like number of filters, kernel size, activation function, etc.
print(layer_new.get_config())

# Print the representation of the newly created convolutional layer.
# This shows the type of layer along with some key configuration parameters in a readable format.
print(layer_new)

# Print the representation of the second layer of the original model for comparison.
# This helps to confirm that the new layer has the same configuration as the model's second layer.
print(model.layers[1])

In [ ]:
#@title Example of Layer Config - New Layer from Current Layer
# Extract the third-to-last layer from the 'model'.
# The 'model' is assumed to be a pre-defined NN model.
# Using a negative index (-3) selects layers from the end of the list.
layer = model.layers[-3]

# Create a new layer 'layer_new' with the same configuration as 'layer'.
# The 'from_config()' method creates a new instance of the layer using the configuration
# dictionary obtained from 'layer.get_config()'. This is useful for duplicating layers.
layer_new = layer.from_config(layer.get_config())

# Print the configuration of the newly created layer.
# The configuration is a dictionary detailing the properties of the layer, like type, size, etc.
print(layer_new.get_config())

# Print the representation of the newly created layer.
# This output typically includes the type of layer & key configuration attributes.
print(layer_new)

# Print the representation of the original layer (third-to-last layer of the model).
# This is useful for comparison to confirm that 'layer_new' has the same configuration as 'layer'.
print(layer)



### **2.2. Model Config**

In [ ]:
#@title Example of Model Config - Create A Sample Model.
'''
Runtime: CPU
tf.keras.Input:                       https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Conv2D:               https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:         https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:              https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.Model:                       https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                   https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.models                       https://www.tensorflow.org/api_docs/python/tf/keras/models
tf.keras.models.clone_model:          https://www.tensorflow.org/api_docs/python/tf/keras/models/clone_model
'''
import tensorflow as tf

# Define the LeNet model architecture using the functional API
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(6, (5, 5))(inputs)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Conv2D(16, (5, 5))(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(120)(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.Dense(84)(x)
x = tf.keras.layers.ReLU()(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="model")

# Compile the model with optimizer, loss function, & metrics
model.compile(optimizer=tf.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()

In [ ]:
#@title Example of Model Config - Clone A Model

# Define a function to extract the configuration for compiling a model
def fun_compile_config(model):
    # Get the optimizer's configuration from the model. The configuration includes all
    # hyperparameters and settings of the optimizer that the model is using.
    optimizer_config = model.optimizer.get_config()

    # Initialize an empty dictionary to store model compile configurations.
    config = {}

    # Store the loss function used by the model in the config dictionary under the key 'loss'.
    config['loss'] = model.loss

    # Store the list of metrics that the model uses during training and evaluation
    # in the config dictionary under the key 'metrics'. The '_metrics' attribute
    # contains the actual metric objects used by the model.
    config['metrics'] = model.compiled_metrics._metrics

    # Create a new optimizer object of the same type as the model's current optimizer,
    # using the previously extracted configuration. This ensures that the optimizer is
    # initialized with the same settings.
    config['optimizer'] = type(model.optimizer).from_config(optimizer_config)

    # Return the dictionary containing the loss function, metrics, and optimizer
    # configuration. This dictionary has all the necessary information to compile
    # a clone of the original model.
    return config

# Clone the model by creating a new, untrained model that has the same architecture
# as the original model. The clone_model function does not copy the weights or the
# configuration for how the model should be compiled.
model_clone = tf.keras.models.clone_model(model)

# Compile the cloned model using the compile configuration of the original model.
# The ** operator is used to unpack the configuration dictionary into the
# compile() method's keyword arguments.
model_clone.compile(**fun_compile_config(model))

# Print the memory locations of the original model & the cloned model.
# This is to show that they are indeed two different instances in memory.
print('Memory Locations')
print(model)
print(model_clone)

# Print an example of the weights from a specific layer of both models to demonstrate
# that they have different weights initially.
# Here, the first weight of the first kernel in the second layer is printed for both models.
print("\nExample of Weights")
print(model.layers[1].weights[0].numpy()[0,0,0,0])
print(model_clone.layers[1].weights[0].numpy()[0,0,0,0])

# Use the set_weights method to copy the weights from the original model ('model')
# to the cloned model ('model_clone'). This effectively trains 'model_clone' with
# the same weights as 'model'.
print("\nUse Set Weights")
model_clone.set_weights(model.get_weights())

# Print the same weight from the same layer of both models again to show that
# after using set_weights, the cloned model now has the same weights as the original.
print(model.layers[1].weights[0].numpy()[0,0,0,0])
print(model_clone.layers[1].weights[0].numpy()[0,0,0,0])

In [ ]:
#@title Example of Model Config - Save & Load Models
# Define file paths for saving the model & its weights.
# The paths specify where the model & its weights will be stored after saving.
path         = "/content/h5_folder/"
path_h5      = "/content/h5_folder/model.keras"  # Path to save the entire model (architecture + weights).
path_weights = "/content/weight_folder/model.weights"  # Path to save only the weights of the model.

# Create the folder using os.mkdir.
import os
if not os.path.exists(path):
    os.mkdir(path)

# Save the entire model to the specified path in HDF5 format (.keras).
# This includes the model's architecture, weights, & training configuration.
model.save(path_h5)

# Save only the weights of the model to the specified path.
# This is useful for cases where you only need to store the trained weights, not the entire model.
model.save_weights(path_weights)

# Print a statement for clarity in the output.
print("Load h5 or Keras models")

# Load the entire model from the saved file.
# This loaded model ('model_loaded') includes the architecture & weights.
model_loaded = tf.keras.models.load_model(path_h5)

# Print a weight from a specific layer of the original & loaded models for comparison.
# This is to verify that the model was saved & loaded correctly, retaining the same weights.
print(model.layers[1].weights[0].numpy()[0,0,0,0])
print(model_loaded.layers[1].weights[0].numpy()[0,0,0,0])

# Print a statement for clarity in the output.
print("Load Weights")

# Create a clone of the original model using tf.keras.models.clone_model.
# This cloned model ('model_clone') will have the same architecture but not the weights.
model_clone = tf.keras.models.clone_model(model)

# Load the weights into the cloned model from the saved weights file.
# This applies the trained weights to the untrained cloned model.
model_clone.load_weights(path_weights)

# Print a weight from a specific layer of the original & cloned models for comparison.
# This confirms that the cloned model now has the same weights as the original after loading.
print(model.layers[1].weights[0].numpy()[0,0,0,0])
print(model_clone.layers[1].weights[0].numpy()[0,0,0,0])


In [ ]:
#@title Example of Model Config - Variables
for var in model.trainable_variables:
    print(var.name, var.shape)

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

## **2. TensorFlow Gradient Tape**

> In TensorFlow, GradientTape is used for automatic differentiation. Automatic differentiation is a key process in training ML models, particularly in NNs, where it is used to compute gradients for optimization. Here are some reasons why GradientTape is specifically used:

> Custom Training Loops: While TensorFlow offers high-level APIs like tf.keras for training models, these APIs abstract away many details of the training process. GradientTape allows for more flexibility & control by enabling custom training loops. This is particularly useful for advanced models or training procedures where you need granular control over the training process.

> Gradients of Arbitrary Computations: GradientTape can compute gradients not just for NN layers but for any differentiable computation. This is useful in research & in the implementation of novel ML algorithms, where you might need gradients for operations that are not standard in NN training.

> Eager Execution: TensorFlow's eager execution mode, which is more intuitive & Pythonic, works seamlessly with GradientTape. Eager execution runs operations immediately & returns their values without building graphs. GradientTape records operations for automatic differentiation during eager execution.

> Complex Models & Loss Functions: When working with complex models (like those with multiple outputs) or custom loss functions, GradientTape provides the flexibility to manually compute gradients. This is particularly important when the automatic gradient computation of standard APIs is not sufficient or needs customization.

> Higher-Order Derivatives: GradientTape can be nested to compute higher-order derivatives, which are derivatives of derivatives. This capability is useful in some advanced ML algorithms & research scenarios.

> Debugging & Education: For learning & debugging purposes, GradientTape offers a clear & explicit way to see & manipulate the gradients. It can be a valuable educational tool for understanding how gradients flow through a model.

In [ ]:
# @title
#@title Example of TensorFlow Gradient Tape
'''
Runtime: GPU $$$
tf.keras.Input:                                 https://www.tensorflow.org/api_docs/python/tf/keras/Input
tf.keras.layers.Conv2D:                        https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
tf.keras.layers.MaxPooling2D:                  https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D
tf.keras.layers.Flatten:                       https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten
tf.keras.layers.Dense:                         https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
tf.keras.Model:                                https://www.tensorflow.org/api_docs/python/tf/keras/Model
tf.optimizers.Adam:                            https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.optimizers.Adam:                      https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam
tf.keras.losses.SparseCategoricalCrossentropy: https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy
tf.function:                                   https://www.tensorflow.org/api_docs/python/tf/function
tf.GradientTape:                               https://www.tensorflow.org/api_docs/python/tf/GradientTape
model.compile:                                 https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile
model.evaluate:                                https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate
model.summary:                                 https://www.tensorflow.org/api_docs/python/tf/keras/Model#summary
'''
import tensorflow as tf

# Load & prepare the MNIST dataset
mnist = tf.keras.datasets.mnist
(datain_tr, dataou_tr), (datain_vl, dataou_vl) = mnist.load_data()

# Normalize the data
datain_tr, datain_vl = datain_tr / 255.0, datain_vl / 255.0

# Reshape data to fit LeNet (adding the channel dimension)
datain_tr = datain_tr[..., tf.newaxis]
datain_vl = datain_vl[..., tf.newaxis]

# Building the LeNet model using the Functional API
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(6, kernel_size=(5, 5), activation='relu')(inputs)
x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Conv2D(16, kernel_size=(5, 5), activation='relu')(x)
x = tf.keras.layers.AveragePooling2D(pool_size=(2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(120, activation='relu')(x)
x = tf.keras.layers.Dense(84, activation='relu')(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='LeNet')

# Define the optimizer & loss function for training the model
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer is a popular choice for training NNs
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()  # Loss function for multi-class classification

# Define a training step function using TensorFlow's Graph execution
@tf.function  # This decorator compiles the function into a callable TensorFlow graph, enhancing performance
def train_step(inputs, targets):
    with tf.GradientTape() as tape:  # Context manager to record operations for automatic differentiation
        predictions = model(inputs, training=True)  # Forward pass: compute predictions from the model
        loss = loss_fn(targets, predictions)  # Compute the loss by comparing predictions to true targets
    gradients = tape.gradient(loss, model.trainable_variables)  # Calculate gradients of loss w.r.t. model's trainable parameters
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))  # Update model parameters based on gradients
    return loss  # Return the computed loss

# Custom training loop
epochs = 3  # Number of epochs (full passes over the training dataset)
for epoch in range(epochs):  # Iterate over epochs
    for step in range(len(datain_tr) // 32):  # Iterate over batches; assuming batch size of 32
        # Extract batches for this step
        batch_inputs = datain_tr[step * 32: (step + 1) * 32]
        batch_targets = dataou_tr[step * 32: (step + 1) * 32]
        loss = train_step(batch_inputs, batch_targets)  # Perform training step & get loss
        # Print loss every 250 steps
        if not step % 250:
            print(f"Epoch {epoch + 1:03d}/{epochs:03d} | Step {step + 1:05d}/{len(datain_tr) // 32} | Loss: {loss.numpy():2.5f}")

# Compile the model for evaluation
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])  # Compile model with same optimizer & loss function, adding accuracy as a metric

# Print a summary of the model
model.summary()  # Displays model architecture including layer types, output shapes & number of parameters

# Evaluate the model on training & validation data
train_loss = model.evaluate(datain_tr, dataou_tr, verbose=0)  # Evaluate on training data
val_loss = model.evaluate(datain_vl, dataou_vl, verbose=0)  # Evaluate on validation data
print("Training Loss: ", train_loss)  # Print training loss
print("Validation Loss: ", val_loss)  # Print validation loss

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


# <font color="#418FDE" size="10" uppercase>**D: TensorFlow Config**</font>
----


In this lecture, you learned to:
* Investigate TensorFlow layers & model configuration information.
* Apply TensorFlow gradient tape to train ML models.

In the next Module (Module 3), we will go over "Representation Learning, Part 1"